# Zermelo Navigation (Continuous)
This notebook provides a detailed, mathematically rigorous walkthrough of the **Holonomic Association Model (HAM)**.
The HAM library treats differentiable **Finsler Geometries** as learnable modules in JAX, allowing us to solve optimal control and simulation problems purely through geometric dynamics.

### Core Philosophy: Metric-First Design
In HAM, the `FinslerMetric` object is the single source of truth. We define the scalar **Energy Functional**:
$$\mathcal{E}[\gamma] = \int \frac{1}{2} F^2(x, \dot{x}) dt$$
where $F(x,v)$ is the Finsler cost function. Everything else—including the Geodesic Spray (the physics engine), curvature, and Berwald parallel transport—is auto-differentiated directly from $F$ using `jax.grad` and `jax.hessian`, entirely avoiding the $O(N^3)$ computational cost of expanding Christoffel symbols manually.

## 1. Setup and Imports
We begin by importing JAX and the HAM library. HAM provides geometric primitives such as topological constraints (`Sphere`) and specific metric classes (`Euclidean`, `Randers`). We also import Plotly for interactive 3D rendering.


In [ ]:
import jax.numpy as jnp
import jax
import plotly.graph_objects as go
import numpy as np

from ham.geometry import Sphere, Randers, TriangularMesh
from ham.geometry import Euclidean
from ham.solvers import AVBDSolver
from ham.vis import generate_icosphere


## 2. Zermelo's Navigation Problem (The Randers Metric)
To model asymmetric travel costs (e.g. wind, ocean currents, or chemical gradients), HAM uses the **Randers Metric**. This is derived from Zermelo's Navigation Problem, which asks for the time-optimal path across a Riemannian manifold (a "sea" defined by $h_{ij}(x)$) subject to a background vector field (a "wind" $W^i(x)$).

The resulting Finsler metric is given by the exact formula:
$$ F(x, v) = \frac{\sqrt{\lambda \|v\|_h^2 + \langle W, v \rangle_h^2} - \langle W, v \rangle_h}{\lambda} $$
where $\lambda = 1 - \|W\|_h^2$. 

HAM strictly enforces the causality constraint $\|W\|_h < 1$ (the wind cannot be faster than the agent's maximum speed).

Below, we define an equatorial trade wind system rotating around the Z-axis.


In [ ]:
# Radius of the continuous manifold
radius = 1.0
sphere_cont = Sphere(radius)

# Wind: Strong rotation around Z-axis
# Strength 0.8 at equator. Counter-Clockwise.
def w_net(x): 
    base = jnp.array([-x[1], x[0], 0.0])
    return 0.9 * base

def h_net(x): 
    return jnp.eye(3)

metric_randers = Randers(sphere_cont, h_net, w_net)
metric_riem = Randers(sphere_cont, h_net, lambda x: jnp.zeros(3))


## 3. Mission: South -> North
We set our start point at the equator and our target destination at the north pole. We aim to compute the geodesic path bridging these points.


In [ ]:
start = jnp.array([1.0, 0.0, 0.0])
end   = jnp.array([0.0,  0.0, 1.0])


## 4. Solving the Boundary Value Problem (AVBD)
To find the geodesics, we use HAM's **Augmented Vertex Block Descent (AVBD)** solver. 
Unlike an Initial Value Problem solver (which shoots a particle and hopes it hits the target), the AVBD solver formulates a Boundary Value Problem (BVP). It optimizes discrete path vertices $x_0, \dots, x_N$ directly to minimize the global energy $\sum E(x_i, v_i)$ subject to topological manifold constraints.

- **Riemannian**: Will yield the standard shortest distance path (a Great Circle).
- **Randers**: Will yield the fastest path taking advantage of the wind.


In [ ]:
solver = AVBDSolver(step_size=0.05, beta=10.0, iterations=200, tol=1e-6)

print("Solving Riemannian (Great Circle)...")
traj_riem = solver.solve(metric_riem, start, end, n_steps=40)

print("Solving Randers (Zermelo)...")
traj_rand = solver.solve(metric_randers, start, end, n_steps=40)

# Calculate Energies
batch_energy = jax.vmap(metric_randers.energy)
e_riem = batch_energy(traj_riem.xs[:-1], traj_riem.vs).sum()
e_rand = batch_energy(traj_rand.xs[:-1], traj_rand.vs).sum()

print(f"Energy Riemannian path: {e_riem:.4f}")
print(f"Energy Randers path:    {e_rand:.4f}")


## 5. Solving on a Discrete Mesh
HAM bridges continuous mathematics with discrete geometry. Here we build an icosphere mesh (`TriangularMesh`) and solve for the discrete geodesic path natively over the facets.


In [ ]:
# High-Res Icosphere (Subdivision=3 -> ~1280 faces)
verts, faces = generate_icosphere(radius=1.0, subdivisions=3)
mesh_discrete = TriangularMesh(verts, faces)
metric_mesh = Euclidean(mesh_discrete) # Isotropic benchmark

print("Solving Discrete Mesh Geodesic...")
traj_mesh = solver.solve(metric_mesh, start, end, n_steps=40)


## 6. Interactive 3D Visualization
We use Plotly to render an interactive 3D visualization. We plot the vector field, the continuous Riemannian path, the continuous Randers path, and the discrete mesh path. You can rotate and zoom to inspect how the Randers path veers off the shortest-distance geodesic to exploit the wind.


In [ ]:
fig = go.Figure()

# Plot transparent sphere (Mesh3d)
fig.add_trace(go.Mesh3d(
    x=verts[:,0], y=verts[:,1], z=verts[:,2],
    i=faces[:,0], j=faces[:,1], k=faces[:,2],
    color='lightblue', opacity=0.3, alphahull=0, name='Manifold'
))

# Plot Wind Vectors (Cones)
theta = np.linspace(0, 2*np.pi, 20)
equator_pts = np.stack([np.cos(theta), np.sin(theta), np.zeros_like(theta)], axis=1)
wind_vecs = np.array(jax.vmap(w_net)(jnp.array(equator_pts)))

fig.add_trace(go.Cone(
    x=equator_pts[:,0], y=equator_pts[:,1], z=equator_pts[:,2],
    u=wind_vecs[:,0], v=wind_vecs[:,1], w=wind_vecs[:,2],
    sizemode='absolute', sizeref=0.2, colorscale='Blues', showscale=False, name='Wind'
))

# Plot Paths
fig.add_trace(go.Scatter3d(
    x=traj_riem.xs[:,0], y=traj_riem.xs[:,1], z=traj_riem.xs[:,2],
    mode='lines', line=dict(color='gray', width=4, dash='dash'), name='Riemannian (Great Circle)'
))
fig.add_trace(go.Scatter3d(
    x=traj_rand.xs[:,0], y=traj_rand.xs[:,1], z=traj_rand.xs[:,2],
    mode='lines', line=dict(color='green', width=6), name='Randers (Wind Optimized)'
))
fig.add_trace(go.Scatter3d(
    x=traj_mesh.xs[:,0], y=traj_mesh.xs[:,1], z=traj_mesh.xs[:,2],
    mode='lines', line=dict(color='orange', width=4, dash='dot'), name='Discrete Mesh Path'
))

fig.update_layout(
    title=f"Zermelo Navigation S^2 | Energy: Randers={e_rand:.2f}, Riem={e_riem:.2f}", 
    scene=dict(aspectmode='data')
)
fig.show()
